# HW5 — Attention Circuits in GPT-2
## Stats 292 — Statistical Models of Text and Language
### Prof. David Donoho · Stanford University · Spring 2026

**Published:** May 29, 2026 · **Due:** June 5, 2026, 11:59 PM PDT

---

## Overview

HW4 put you inside the residual stream: you read it at every layer with the logit lens,
extracted a linear direction for factual truth, and surgically removed it. You treated
attention heads as a black box.

This assignment opens the box. You will analyze the *circuit structure* of GPT-2's
attention heads: how individual heads transform the residual stream, how they detect
patterns in sequences, and how two heads in different layers can **compose** — the
output of one shaping the behavior of the other — to implement a concrete algorithm.

The example you will reconstruct is the **induction circuit**: a two-head mechanism
that predicts repeated tokens. By the end you will have found it, verified it causally,
and understood *why* two heads are necessary.

**Reading required before starting:**
- Elhage et al. (2021), *A Mathematical Framework for Transformer Circuits* — Sections 1–4
- ch10 deck — OV/QK virtual weights, residual stream as linear communication channel


## Setup


In [ ]:
# Environment: conda activate stats292
# (transformer_lens, einops, datasets are in environment.yml)
# If running on Colab instead:
# !pip install -q transformer_lens einops datasets


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import einops
import torch.nn.functional as F
from transformer_lens.model_bridge import TransformerBridge

bridge = TransformerBridge.boot_transformers('gpt2', device='cpu')
bridge.enable_compatibility_mode()   # fold LN, center weights

n_layers = bridge.cfg.n_layers   # 12
n_heads  = bridge.cfg.n_heads    # 12
d_model  = bridge.cfg.d_model    # 768
d_head   = bridge.cfg.d_head     # 64
d_vocab  = bridge.cfg.d_vocab    # 50257

print(f'GPT-2 small: {n_layers} layers, {n_heads} heads/layer, '
      f'd_model={d_model}, d_head={d_head}, d_vocab={d_vocab}')


---
## Part 1 — OV and QK Circuits (~2 hours)

Every attention head defines two linear maps on the residual stream:

$$W_{OV}^{(h)} = W_V^{(h)} W_O^{(h)} \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}$$

$$W_{QK}^{(h)} = W_Q^{(h)} W_K^{(h)\top} \in \mathbb{R}^{d_{\text{model}} \times d_{\text{model}}}$$

$W_{OV}$ describes what the head *writes* to the residual stream given the token it attends to.
$W_{QK}$ describes which tokens attend to which.

Weight shapes in TransformerLens (GPT-2 small):

| Tensor | Shape |
|--------|-------|
| `bridge.W_Q`, `bridge.W_K`, `bridge.W_V` | `(12, 12, 768, 64)` — layers × heads × d\_model × d\_head |
| `bridge.W_O` | `(12, 12, 64, 768)` — layers × heads × d\_head × d\_model |
| `bridge.W_E` | `(50257, 768)` — token embedding |
| `bridge.W_U` | `(768, 50257)` — unembedding |


In [ ]:
# 1.1 — Model dimensions
# The rank of W_OV = W_V @ W_O is at most d_head (= 64), since W_V: d_model→d_head
# and W_O: d_head→d_model, so the composition passes through a d_head-dimensional bottleneck.

print(f'd_model={d_model}, n_layers={n_layers}, n_heads={n_heads}, d_head={d_head}')

# TODO (1.1): What does rank d_head=64 imply about how much information one head
# can write per token, given d_model=768?
# Write your answer as a comment or markdown cell below.


**Answer 1.1:** *(your explanation here)*


In [ ]:
# 1.2 — OV circuit and copy behavior
W_E = bridge.W_E   # (d_vocab, d_model)
W_U = bridge.W_U   # (d_model, d_vocab)

layer = 0
W_OV_all = []

for h in range(n_heads):
    W_V_h = bridge.W_V[layer, h]  # (d_model, d_head)
    W_O_h = bridge.W_O[layer, h]  # (d_head, d_model)
    # TODO: compute W_OV_h = W_V_h @ W_O_h
    W_OV_h = None  # replace
    W_OV_all.append(W_OV_h)

# TODO: for each head compute full_circuit = W_U @ W_OV_h @ W_E.T — shape (d_vocab, d_vocab)
# Examine the diagonal: which heads have the largest diagonal values (copy heads)?
# Report the Frobenius norm of each W_OV_h and identify the top-5 copy heads.


**Answer 1.2:** *(which heads copy? what does the diagonal tell you?)*


In [ ]:
# 1.3 — QK circuit
W_QK_all = []

for h in range(n_heads):
    W_Q_h = bridge.W_Q[layer, h]  # (d_model, d_head)
    W_K_h = bridge.W_K[layer, h]  # (d_model, d_head)
    # TODO: compute W_QK_h = W_Q_h @ W_K_h.T  — shape (d_model, d_model)
    W_QK_h = None  # replace
    W_QK_all.append(W_QK_h)

# For each head compute token_attn = W_E @ W_QK_h @ W_E.T — shape (d_vocab, d_vocab)
# Report mean and std of the diagonal vs off-diagonal entries.
# Do any layer-0 heads attend preferentially to tokens of the same type?


**Answer 1.3:** *(diagonal vs off-diagonal statistics; which heads, if any, have large diagonal?)*

**Answer 1.4 (Mathematical):** $W_{QK}$ is a bilinear form and is generally not symmetric.
What does asymmetry mean mechanistically — if $x^\top W_{QK} y \neq y^\top W_{QK} x$, what
does this say about the relationship between attending and being attended to?
Is it possible for a head to implement a *symmetric* attention pattern from an *asymmetric* $W_{QK}$?

*(your answer here)*


---
## Part 2 — Finding Induction Heads (~3 hours)

An **induction head** implements: find the most recent previous occurrence of the current
token, then predict that whatever followed it will follow again.
On a repeated sequence $[t_1,\ldots,t_N,t_1,\ldots,t_N]$, when processing position $N+i$,
an induction head attends strongly to position $i$ (one after the previous occurrence of $t_i$).

The **induction score** of head $(\ell, h)$ on a repeated sequence of length $2N$ is:

$$\text{IS}(\ell, h) = \frac{1}{N} \sum_{i=1}^{N} A_{\ell,h}[N+i,\; i]$$

where $A_{\ell,h}[p,q]$ is the attention weight from position $p$ to position $q$.


In [ ]:
# 2.1 — Build a repeated random-token sequence
torch.manual_seed(42)
N = 50
tokens_half = torch.randint(0, d_vocab, (1, N))
repeated    = torch.cat([tokens_half, tokens_half], dim=1)  # shape (1, 2N)

print(f'Sequence length: {repeated.shape[1]} (N={N}, repeated twice)')

logits, cache = bridge.run_with_cache(repeated)
print('Cache keys available (sample):', list(cache.keys())[:6])


In [ ]:
# 2.2 — Compute induction scores for all heads
induction_scores = torch.zeros(n_layers, n_heads)

for layer_idx in range(n_layers):
    # Attention pattern shape: (batch, n_heads, seq, seq)
    attn = cache['pattern', layer_idx][0]  # (n_heads, 2N, 2N)
    for h in range(n_heads):
        # TODO: IS(layer_idx, h) = mean of attn[h, N+i, i] for i in range(N)
        # attn[h, N+i, i] is the weight position N+i pays to position i
        # (one after the previous occurrence of the token at position N+i)
        pass

top_l, top_h = divmod(induction_scores.argmax().item(), n_heads)
print(f'Max induction score: {induction_scores.max():.3f}  at Layer {top_l}, Head {top_h}')


In [ ]:
THRESHOLD = 0.4
induction_heads = [(l, h) for l in range(n_layers) for h in range(n_heads)
                   if induction_scores[l, h] > THRESHOLD]
print(f'Induction heads (IS > {THRESHOLD}):', induction_heads)

# Fallback so downstream cells don't crash before 2.2 is complete
if induction_heads:
    top_layer, top_head = max(induction_heads, key=lambda lh: induction_scores[lh[0], lh[1]])
else:
    top_layer, top_head = 5, 1  # placeholder — fill in 2.2 first
    print(f'(Using placeholder top_layer={top_layer}, top_head={top_head} until 2.2 is complete)')


**Answer 2.2:** *(which layers? does this match the expected answer of layers 5–6 for GPT-2 small?)*


In [ ]:
# 2.3 — Visualize the top induction head's attention pattern
attn_pattern = cache['pattern', top_layer][0, top_head].detach().numpy()  # (2N, 2N)

fig, ax = plt.subplots(figsize=(8, 8))
sns.heatmap(attn_pattern, cmap='Blues', ax=ax)
ax.axhline(N, color='red', linewidth=1.5, linestyle='--')
ax.axvline(N, color='red', linewidth=1.5, linestyle='--')
ax.set_title(f'Attention: Layer {top_layer}, Head {top_head}  (IS={induction_scores[top_layer, top_head]:.3f})')
ax.set_xlabel('Key position'); ax.set_ylabel('Query position')
plt.tight_layout(); plt.show()


**Answer 2.3:** *(describe the pattern in the lower-left block vs. the lower-right block)*


In [ ]:
# 2.4 — Stability across different random sequences
n_trials = 10
scores_per_trial = torch.zeros(n_trials, n_layers, n_heads)

for trial in range(n_trials):
    toks = torch.randint(0, d_vocab, (1, N))
    rep  = torch.cat([toks, toks], dim=1)
    _, c = bridge.run_with_cache(rep)
    for li in range(n_layers):
        attn_t = c['pattern', li][0]
        for h in range(n_heads):
            # TODO: compute IS for this trial
            pass

# Report mean and std for the top-3 induction heads
top3 = sorted(induction_heads, key=lambda lh: induction_scores[lh[0],lh[1]], reverse=True)[:3]
print('Stability of top-3 induction heads across 10 random sequences:')
for (l, h) in top3:
    s = scores_per_trial[:, l, h]
    print(f'  L{l}H{h}: mean={s.mean():.3f}  std={s.std():.3f}')


**Answer 2.4:** *(is the induction score stable? what does stability imply about the mechanism?)*

**Answer 2.5 (Mathematical):** An induction head attends to position $i$ when processing
position $N+i$ because $t_i = t_{N+i}$. But the attention pattern is determined by
$x_{N+i}^\top W_{QK} x_i$, not token identity directly. What must be true about $W_{QK}$
for this to work? What relationship between the query and key representations of the
*same* token makes the dot product large?

*(your answer here)*


---
## Part 3 — Causal Ablation (~3 hours)

The induction score identifies *correlational* evidence. To confirm causality, we
**ablate**: silence the heads and measure whether induction performance drops
specifically on sequences that require the induction mechanism.

In TransformerLens, the per-head output after the $W_O$ projection is at
`blocks.{layer}.attn.hook_result` — shape `(batch, seq, n_heads, d_model)`.
Zeroing the $h$-th slice removes that head's contribution to the residual stream.


In [ ]:
# 3.1 — Ablation hook
def make_ablation_hook(head_idx):
    """Returns a hook that zeros out the output of head `head_idx`."""
    def hook_fn(value, hook):
        # value shape: (batch, seq, n_heads, d_model)
        value[:, :, head_idx, :] = 0.0
        return value
    return hook_fn

def hook_name(layer_idx):
    return f'blocks.{layer_idx}.attn.hook_result'

print('Ablation utilities defined.')
print(f'Will ablate: Layer {top_layer}, Head {top_head}')


In [ ]:
# 3.2 — Helper: per-token loss on the second half of a repeated sequence
def loss_second_half(toks_1d, fwd_hooks=None):
    """Cross-entropy loss averaged over positions N to 2N-1."""
    toks_2d = toks_1d.unsqueeze(0)  # (1, 2N)
    if fwd_hooks:
        logits = bridge.run_with_hooks(toks_2d, fwd_hooks=fwd_hooks)
    else:
        logits = bridge(toks_2d)
    log_probs = F.log_softmax(logits[0, N-1:2*N-1], dim=-1)  # predict positions N..2N-1
    targets = toks_1d[N:2*N]
    return -log_probs[range(N), targets].mean().item()

# 3.2 — Repeated vs random: measure ablation effect
n_trials = 20
ablation_hooks = [(hook_name(top_layer), make_ablation_hook(top_head))]

delta_repeated = []
delta_random   = []

for _ in range(n_trials):
    half  = torch.randint(0, d_vocab, (N,))
    rep   = torch.cat([half, half])           # repeated sequence
    rand  = torch.randint(0, d_vocab, (2*N,)) # random (no structure)

    # TODO: compute loss_second_half(rep) and loss_second_half(rep, fwd_hooks=ablation_hooks)
    # then delta = ablated_loss - baseline_loss
    # do the same for rand
    pass

print(f'Mean loss increase on REPEATED sequences: {np.mean(delta_repeated):.4f}')
print(f'Mean loss increase on RANDOM   sequences: {np.mean(delta_random):.4f}')


**Answer 3.2:** *(is the ablation effect specific to repeated sequences? what does this imply?)*


In [ ]:
# 3.3 — Individual ablations of top-5 induction heads
top5 = sorted(induction_heads, key=lambda lh: induction_scores[lh[0],lh[1]], reverse=True)[:5]
print('Top-5 induction heads:', top5)

# TODO: for each head in top5, compute mean loss increase on 20 repeated sequences
# Report: which heads matter most? Are the effects proportional to induction score?


**Answer 3.3:** *(individual effect sizes; do all induction heads contribute equally?)*


In [ ]:
# 3.4 — Joint ablation of all top-5 heads
# TODO: build a hooks list that ablates all top-5 heads simultaneously
# Measure mean loss increase on 20 repeated sequences
# Compare: joint_effect vs sum_of_individual_effects
joint_hooks = []  # TODO: populate


**Answer 3.4:** *(is the joint effect larger or smaller than the sum of individual effects?
what does this imply about whether the heads are redundant or complementary?)*

**Answer 3.5 (Mathematical):** Our ablation zeros the head's output. A softer intervention
is *mean ablation*: replace with the average output over many inputs rather than zero.
Why might mean ablation be preferred? Under what assumption are the two equivalent?

*(your answer here)*


---
## Part 4 — Composition Scores (~3 hours)

Induction heads cannot operate alone. A single-layer induction head fails because
at layer 1 the residual stream contains only the token embedding — it does not yet
contain information about which tokens *preceded* the current one.

The two-head induction circuit: (1) a **previous-token head** at layer 0 attends to
position $i-1$ and writes information about that token into the stream; (2) an
**induction head** at layer 1 reads this via its *key* projection.

The **K-composition score** measures how much head $B$'s key subspace overlaps with
head $A$'s OV output subspace:

$$\text{comp}_K(A,B) = \frac{\|W_{OV}^A \cdot W_K^B\|_F}{\|W_{OV}^A\|_F \cdot \|W_K^B\|_F}$$

Analogous Q-composition and V-composition scores use $W_Q^B$ and $W_V^B$ respectively.


In [ ]:
# 4.1 — Find previous-token heads in layer 0
def prev_token_score(layer_idx, head_idx, n_sentences=5):
    """Average attention weight on position i-1 when processing position i."""
    sentences = [
        'The quick brown fox jumps over the lazy dog.',
        'Statistics is the science of learning from data.',
        'Language models predict the next token from context.',
        'Attention heads implement linear operations on embeddings.',
        'The residual stream carries information across layers.',
    ]
    scores = []
    for sent in sentences[:n_sentences]:
        toks = bridge.to_tokens(sent)
        _, c = bridge.run_with_cache(toks)
        attn = c['pattern', layer_idx][0, head_idx]  # (seq, seq)
        seq_len = attn.shape[0]
        scores += [attn[i, i-1].item() for i in range(1, seq_len)]
    return float(np.mean(scores))

pt_scores = torch.tensor([prev_token_score(0, h) for h in range(n_heads)])
print('Previous-token scores (layer 0):')
for h in range(n_heads):
    print(f'  Head {h:2d}: {pt_scores[h]:.3f}')

prev_head = pt_scores.argmax().item()
print(f'\nTop previous-token head: Layer 0, Head {prev_head}  (score={pt_scores[prev_head]:.3f})')


**Answer 4.1:** *(which heads score highest? does the top previous-token head compose with the top induction head?)*


In [ ]:
# 4.2 — K-composition scores
def k_comp(la, ha, lb, hb):
    """K-composition score: ||W_OV^A @ W_K^B||_F / (||W_OV^A||_F * ||W_K^B||_F)"""
    W_OV_a = bridge.W_V[la, ha] @ bridge.W_O[la, ha]  # (d_model, d_model)
    W_K_b  = bridge.W_K[lb, hb]                       # (d_model, d_head)
    # TODO: compute comp_K
    numerator   = None  # replace: torch.linalg.norm(W_OV_a @ W_K_b, 'fro')
    denominator = None  # replace: torch.linalg.norm(W_OV_a, 'fro') * torch.linalg.norm(W_K_b, 'fro')
    if numerator is None or denominator is None:
        return 0.0  # complete the TODO above
    return (numerator / denominator).item()

# K-composition: all layer-0 heads vs top induction head
k_scores = torch.tensor([k_comp(0, h0, top_layer, top_head) for h0 in range(n_heads)])
print(f'K-composition scores: layer-0 heads -> L{top_layer}H{top_head}')
for h in range(n_heads):
    print(f'  L0H{h:2d}: {k_scores[h]:.4f}')

best_composing = k_scores.argmax().item()
print(f'\nBest K-composing head: L0H{best_composing}  (score={k_scores[best_composing]:.4f})')


In [ ]:
# 4.3 — Q and V composition scores
def q_comp(la, ha, lb, hb):
    # TODO: ||W_OV^A @ W_Q^B||_F / (||W_OV^A||_F * ||W_Q^B||_F)
    pass

def v_comp(la, ha, lb, hb):
    # TODO: ||W_OV^A @ W_V^B||_F / (||W_OV^A||_F * ||W_V^B||_F)
    pass

# Compare K, Q, V composition for the best-composing pair
la, ha, lb, hb = 0, best_composing, top_layer, top_head
print(f'Composition scores for L{la}H{ha} -> L{lb}H{hb}:')
print(f'  K-composition: {k_comp(la, ha, lb, hb):.4f}')
print(f'  Q-composition: {q_comp(la, ha, lb, hb)}')
print(f'  V-composition: {v_comp(la, ha, lb, hb)}')


**Answer 4.3:** *(which type of composition dominates? why does K-composition make sense for the induction circuit?)*


In [ ]:
# 4.4 — Causal verification: ablate the upstream previous-token head
# Ablate L0H{best_composing} -- NOT the induction head -- and measure effect on repeated sequences

upstream_hooks = [(hook_name(0), make_ablation_hook(best_composing))]

delta_upstream = []
for _ in range(20):
    half = torch.randint(0, d_vocab, (N,))
    rep  = torch.cat([half, half])
    # TODO: compute loss increase when ablating the upstream head only
    pass

print(f'Loss increase ablating UPSTREAM head L0H{best_composing}: {np.mean(delta_upstream):.4f}')
print(f'(Compare to direct induction-head ablation from Part 3)')


**Answer 4.4:** *(does ablating the upstream head disrupt the induction head's performance?)
What does this tell us about the dependency structure of the circuit?*

**Answer 4.5 (Mathematical):**
The K-composition score uses the Frobenius norm. The numerator
$\|W_{OV}^A W_K^B\|_F$ measures how much of $W_K^B$'s input lies in the
column space of $W_{OV}^A$. Show that if $W_{OV}^A$ and $W_K^B$ have
orthogonal column spaces the score is 0. What is the maximum possible value,
and when is it achieved? Relate this to the rank constraint on $W_{OV}^A$ from Part 1.

*(your answer here)*


---
## Part 5 — Extension: The IOI Circuit (~4 hours)

The **Indirect Object Identification** task: given
> *"When Mary and John went to the store, John gave a bottle of milk to"*

the correct next token is *" Mary"* — the name that appeared earlier but was
*not* the most recent subject. Wang et al. (2022) identified a ~26-head circuit
in GPT-2 small: name-mover heads, duplicate-token heads, inhibition heads.


In [ ]:
# 5.1 — IOI dataset and baseline accuracy
# TransformerLens has built-in IOI utilities:
# See: https://github.com/TransformerLensOrg/TransformerLens/blob/main/demos/ioi_demo.ipynb

# TODO: construct 100 IOI prompts and verify GPT-2 small predicts the correct
# indirect object more often than the repeated subject name.
# Report: fraction correct.


In [ ]:
# 5.2 — Path patching to find important heads
# TODO: use TransformerLens path_patching to measure each head's contribution
# to the IO - S logit difference.
# Report the top-5 most important heads.


In [ ]:
# 5.3 — Categorize heads by function
# TODO: classify your top heads as name-mover, S-inhibition, or duplicate-token
# using the criteria from Wang et al. (2022).
# Compare to the published circuit.


**Answer 5.3:** *(which heads match the published circuit? where do you agree / disagree?)*

**Answer 5.4 (Open question):** The induction circuit has 2 heads; the IOI circuit
has ~26. Both live in GPT-2 small (144 heads total). What does the existence of many
overlapping circuits in the same model imply about superposition in attention?
Can the same head participate in multiple circuits? Give one example from your analysis.

*(your answer here)*


---
## Submission

Submit this notebook with all code cells executed and all questions answered inline.
Figures should be labeled. Mathematical derivations may be in Markdown cells.

**Approximate time:** Parts 1–4 ≈ 11 hours; Part 5 (extension) adds ~4 hours.

---

## Mathematical Reference

| Symbol | Meaning |
|--------|---------|
| $d_{\text{model}}=768$, $L=12$, $H=12$, $d_h=64$, $V=50257$ | GPT-2 small |
| $W_{OV}^{(h)} = W_V^{(h)} W_O^{(h)}$ | OV circuit — rank $d_h$ |
| $W_{QK}^{(h)} = W_Q^{(h)} W_K^{(h)\top}$ | QK circuit |
| $\text{IS}(\ell,h)$ | Induction score |
| $\text{comp}_K(A,B) = \|W_{OV}^A W_K^B\|_F / (\|W_{OV}^A\|_F \|W_K^B\|_F)$ | K-composition |

## References

- Elhage et al. (2021). *A Mathematical Framework for Transformer Circuits.* Transformer Circuits Thread.
- Wang et al. (2022). *Interpretability in the Wild: a Circuit for Indirect Object Identification.* arXiv:2211.00593.
- Nanda & Bloom (2022). *TransformerLens.* github.com/TransformerLensOrg/TransformerLens.
